#### Imports

In [1]:
import torch
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

#### Hyperparameters

In [ ]:
config = dict(
        dataset = '5a',
        kernel = 'linear',
        C = 0.1,
        gamma = 'auto',
        degree = 2
)

#### Util Functions

In [3]:
def to_numpy(tensor: torch.Tensor) -> np.ndarray:
    return tensor.detach().cpu().numpy()

In [4]:
def build_pipeline(config):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(
            kernel=config['kernel'],
            C=config['C'],
            gamma=config['gamma']
            # degree=config['degree'],
        ))
    ])

#### Dataset

In [ ]:
def get_decision(dataset):

    if dataset == '5a':
        path = './data/level_5a.pth'
    # elif dataset == '5b':
    #     path = './data/level_5b_10n.pth'
    # elif dataset == '5c':
    #     path = './data/level_5c_20n.pth'  
    # elif dataset == '5ld':
    #     path = './data/level_5ld.pth'
    # elif dataset == '4a':
    #     path = './data/level_4a.pth'
    else:
        raise ValueError(f"Dataset '{dataset}' does not exist.")
    
    # Load the dataset
    loaded_data = torch.load(path)

    # Access the components
    X = loaded_data["data"]
    y = loaded_data["labels"]
    tree_planes = loaded_data["tree_planes"]
    num_data = X.shape[0]
    num_data = len(X)
    num_train= num_data//2
    num_vali = num_data//4
    num_test = num_data//4
    train_data = X[:num_train,:]
    train_data_labels = y[:num_train]

    vali_data = X[num_train:num_train+num_vali,:]
    vali_data_labels = y[num_train:num_train+num_vali]

    test_data = X[num_train+num_vali :,:]
    test_data_labels = y[num_train+num_vali :]

    if (dataset == '5b') or (dataset == '5c'):
        shuffled_indices = torch.randperm(train_data.shape[0])
        train_data = train_data[shuffled_indices]
        train_data_labels = train_data_labels[shuffled_indices]

    return train_data, train_data_labels, vali_data, vali_data_labels, test_data, test_data_labels, tree_planes

In [6]:

def load_data(dataset):
    train_X, train_y, val_X, val_y, test_X, test_y, _ = get_decision(config['dataset'])
    return (to_numpy(train_X), to_numpy(train_y),
            to_numpy(val_X),   to_numpy(val_y),
            to_numpy(test_X),  to_numpy(test_y))

#### Train

In [7]:
def train_and_evaluate():
    
    # Load data
    X_train, y_train, X_val, y_val, X_test, y_test = load_data(config['dataset'])

    # Build model pipeline
    pipeline = build_pipeline(config)

    # Train
    pipeline.fit(X_train, y_train)
    y_pred_train = pipeline.predict(X_train)
    acc_train = accuracy_score(y_train, y_pred_train)

    # Validate
    y_pred_val = pipeline.predict(X_val)
    acc_val = accuracy_score(y_val, y_pred_val)

    print(f"Train Accuracy : {acc_train}, Validation Accuracy : {acc_val}")

In [8]:
train_and_evaluate()

Train Accuracy : 0.615275, Validation Accuracy : 0.61015
